In [108]:
##Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os


In [109]:
##read data
df = pd.read_csv("../data/raw/ML-EdgeIIoT-dataset.csv")
print(df.shape)

(157800, 63)


C:\Users\12985709\AppData\Local\Temp\ipykernel_29832\3243518439.py:2: DtypeWarning: Columns (3,6,11,13,14,15,16,17,31,32,34,39,45,51,54,55) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/ML-EdgeIIoT-dataset.csv")


In [110]:
##basic info
##df.head()
df.info() #non-null collums
df.columns.tolist()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157800 entries, 0 to 157799
Data columns (total 63 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   frame.time                 157800 non-null  object 
 1   ip.src_host                157800 non-null  object 
 2   ip.dst_host                157800 non-null  object 
 3   arp.dst.proto_ipv4         157800 non-null  object 
 4   arp.opcode                 157800 non-null  float64
 5   arp.hw.size                157800 non-null  float64
 6   arp.src.proto_ipv4         157800 non-null  object 
 7   icmp.checksum              157800 non-null  float64
 8   icmp.seq_le                157800 non-null  float64
 9   icmp.transmit_timestamp    157800 non-null  float64
 10  icmp.unused                157800 non-null  float64
 11  http.file_data             157800 non-null  object 
 12  http.content_length        157800 non-null  float64
 13  http.request.uri.query     15

['frame.time',
 'ip.src_host',
 'ip.dst_host',
 'arp.dst.proto_ipv4',
 'arp.opcode',
 'arp.hw.size',
 'arp.src.proto_ipv4',
 'icmp.checksum',
 'icmp.seq_le',
 'icmp.transmit_timestamp',
 'icmp.unused',
 'http.file_data',
 'http.content_length',
 'http.request.uri.query',
 'http.request.method',
 'http.referer',
 'http.request.full_uri',
 'http.request.version',
 'http.response',
 'http.tls_port',
 'tcp.ack',
 'tcp.ack_raw',
 'tcp.checksum',
 'tcp.connection.fin',
 'tcp.connection.rst',
 'tcp.connection.syn',
 'tcp.connection.synack',
 'tcp.dstport',
 'tcp.flags',
 'tcp.flags.ack',
 'tcp.len',
 'tcp.options',
 'tcp.payload',
 'tcp.seq',
 'tcp.srcport',
 'udp.port',
 'udp.stream',
 'udp.time_delta',
 'dns.qry.name',
 'dns.qry.name.len',
 'dns.qry.qu',
 'dns.qry.type',
 'dns.retransmission',
 'dns.retransmit_request',
 'dns.retransmit_request_in',
 'mqtt.conack.flags',
 'mqtt.conflag.cleansess',
 'mqtt.conflags',
 'mqtt.hdrflags',
 'mqtt.len',
 'mqtt.msg_decoded_as',
 'mqtt.msg',
 'mqtt.m

In [111]:
print(df["Attack_label"].value_counts()) #attack labels

Attack_label
1    133499
0     24301
Name: count, dtype: int64


In [112]:
print(df["Attack_type"].value_counts()) #attack samples

Attack_type
Normal                   24301
DDoS_UDP                 14498
DDoS_ICMP                14090
Ransomware               10925
DDoS_HTTP                10561
SQL_injection            10311
Uploading                10269
DDoS_TCP                 10247
Backdoor                 10195
Vulnerability_scanner    10076
Port_Scanning            10071
XSS                      10052
Password                  9989
MITM                      1214
Fingerprinting            1001
Name: count, dtype: int64


In [113]:
print(df["Attack_type"].nunique()) #number of attack types

15


In [114]:
df.select_dtypes(include="object").columns.tolist() #20个objects字段

['frame.time',
 'ip.src_host',
 'ip.dst_host',
 'arp.dst.proto_ipv4',
 'arp.src.proto_ipv4',
 'http.file_data',
 'http.request.uri.query',
 'http.request.method',
 'http.referer',
 'http.request.full_uri',
 'http.request.version',
 'tcp.options',
 'tcp.payload',
 'tcp.srcport',
 'dns.qry.name.len',
 'mqtt.conack.flags',
 'mqtt.msg',
 'mqtt.protoname',
 'mqtt.topic',
 'Attack_type']

In [115]:
a=df.duplicated().sum()
constant_cols = []

for col in df.columns:
    
    if df[col].nunique() == 1:
        constant_cols.append(col)

cat_cols = df.select_dtypes(
    include=["object"]
).columns.tolist()

print("a=",a)
print("constant_cols:",constant_cols)
print("categorical_cols:",cat_cols)
for col in [
    "tcp.srcport",
    "dns.qry.name.len"
]:
    print("\nCOLUMN:", col)
    print(df[col].head())
    print(df[col].nunique())

a= 814
constant_cols: ['icmp.unused', 'http.tls_port', 'dns.qry.type', 'dns.retransmit_request_in', 'mqtt.msg_decoded_as', 'mbtcp.len', 'mbtcp.trans_id', 'mbtcp.unit_id']
categorical_cols: ['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.src.proto_ipv4', 'http.file_data', 'http.request.uri.query', 'http.request.method', 'http.referer', 'http.request.full_uri', 'http.request.version', 'tcp.options', 'tcp.payload', 'tcp.srcport', 'dns.qry.name.len', 'mqtt.conack.flags', 'mqtt.msg', 'mqtt.protoname', 'mqtt.topic', 'Attack_type']

COLUMN: tcp.srcport
0    0
1    0
2    0
3    0
4    0
Name: tcp.srcport, dtype: object
33551

COLUMN: dns.qry.name.len
0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: dns.qry.name.len, dtype: object
9


In [116]:
df_clean = df.copy()
df_clean = df_clean.drop_duplicates()
#drop leakage columns
leakage_cols = [
    'frame.time',
    'ip.src_host',
    'ip.dst_host'
]
df_clean = df_clean.drop(
    columns=leakage_cols
)
#drop text columns (NLP features)
text_cols = [
    'http.file_data',
    'http.request.uri.query',
    'http.referer',
    'http.request.full_uri',
    'tcp.payload',
    'mqtt.msg'
]
df_clean = df_clean.drop(
    columns=text_cols
)
#drop constant columns
constant_cols = [
    'icmp.unused',
    'http.tls_port',
    'dns.qry.type',
    'dns.retransmit_request_in',
    'mqtt.msg_decoded_as',
    'mbtcp.len',
    'mbtcp.trans_id',
    'mbtcp.unit_id'
]
df_clean = df_clean.drop(
    columns=constant_cols
)



In [117]:
df_clean.shape

(156986, 46)

In [118]:
non_numeric = df_clean[
    ~df_clean["tcp.srcport"].astype(str).str.replace(".0","",regex=False).str.isnumeric()
]

print(non_numeric.shape)

non_numeric["tcp.srcport"].unique()[:50]

(367, 46)


array(['DESKTOP-UHF0SF2', 'DESKTOP-UHF0SF2.local',
       '_googlecast._tcp.local'], dtype=object)

In [119]:
df_clean["tcp.srcport"].value_counts().head(20)

tcp.srcport
80.0                      33569
0.0                       17641
1883.0                    10075
4321.0                     4864
4321.0                     4796
47924.0                    3002
5900.0                     2841
60210.0                    2555
56322.0                    2082
0.0                        1969
3333.0                     1363
35306.0                    1304
60944.0                    1221
80.0                        868
3333.0                      626
_googlecast._tcp.local      216
65535.0                     135
DESKTOP-UHF0SF2.local        77
DESKTOP-UHF0SF2              74
62302.0                      42
Name: count, dtype: int64

In [120]:
# deal with tcp.srcport for abnormalities
mask = (
    ~df_clean["tcp.srcport"].isin([
        "_googlecast._tcp.local",
        "DESKTOP-UHF0SF2",
        "DESKTOP-UHF0SF2.local"
    ])
)

df_clean = df_clean[mask]

df_clean["tcp.srcport"] = pd.to_numeric(
    df_clean["tcp.srcport"],
    errors="coerce"
)

df_clean["tcp.srcport"].dtype

dtype('float64')

In [121]:
#check objective columns
cat_cols = df_clean.select_dtypes(
    include=["object"]
).columns.tolist()

print(cat_cols)

['arp.dst.proto_ipv4', 'arp.src.proto_ipv4', 'http.request.method', 'http.request.version', 'tcp.options', 'dns.qry.name.len', 'mqtt.conack.flags', 'mqtt.protoname', 'mqtt.topic', 'Attack_type']


In [122]:
df_clean["dns.qry.name.len"] = pd.to_numeric(
    df_clean["dns.qry.name.len"],
    errors="coerce"
)
df_clean["dns.qry.name.len"].dtype

dtype('float64')

In [123]:
## DATA Review

cat_cols = [
    'arp.dst.proto_ipv4',
    'arp.src.proto_ipv4',
    'http.request.method',
    'http.request.version',
    'tcp.options',
    'mqtt.conack.flags',
    'mqtt.protoname',
    'mqtt.topic'
]

for col in cat_cols:
    print("="*50)
    print(col)
    print("Unique:", df_clean[col].nunique())
    print(df_clean[col].value_counts().head(10))

arp.dst.proto_ipv4
Unique: 9
arp.dst.proto_ipv4
0                126882
0                 26728
192.168.0.128      2052
192.168.0.170       595
192.168.0.147       223
192.168.0.1          87
0.0                  33
192.168.0.101        12
192.168.0.129         7
Name: count, dtype: int64
arp.src.proto_ipv4
Unique: 9
arp.src.proto_ipv4
0                110040
0                 43112
0.0                1893
192.168.0.128       683
192.168.0.170       562
192.168.0.1         311
192.168.0.101        16
0.0.0.0               1
192.168.0.129         1
Name: count, dtype: int64
http.request.method
Unique: 7
http.request.method
0          54062
0.0        52249
0.0        43112
GET         6676
POST         267
TRACE        252
OPTIONS        1
Name: count, dtype: int64
http.request.version
Unique: 9
http.request.version
0                                                          54095
0.0                                                        52216
0.0                                        

In [124]:
#sort http.request.method category
sorted(
    df_clean["http.request.method"]
    .astype(str)
    .unique()
)

['0', '0.0', 'GET', 'OPTIONS', 'POST', 'TRACE']

In [125]:
categorical_cols = [
    'arp.dst.proto_ipv4',
    'arp.src.proto_ipv4',
    'http.request.method',
    'http.request.version',
    'mqtt.conack.flags',
    'mqtt.protoname',
    'mqtt.topic'
]

for col in categorical_cols:

    df_clean[col] = (
        df_clean[col]
        .astype(str)
        .str.strip()
        .replace({
            "0.0": "0"
        })
    )

In [126]:
#tco.options has 73k unique values, which is too high for a categorical feature, and it is not important for attack detection, so we drop it.
df_clean = df_clean.drop(
    columns=["tcp.options"]
)

In [127]:
for col in categorical_cols:

    print("="*50)

    print(col)

    print(df_clean[col].value_counts())

arp.dst.proto_ipv4
arp.dst.proto_ipv4
0                153643
192.168.0.128      2052
192.168.0.170       595
192.168.0.147       223
192.168.0.1          87
192.168.0.101        12
192.168.0.129         7
Name: count, dtype: int64
arp.src.proto_ipv4
arp.src.proto_ipv4
0                155045
192.168.0.128       683
192.168.0.170       562
192.168.0.1         311
192.168.0.101        16
0.0.0.0               1
192.168.0.129         1
Name: count, dtype: int64
http.request.method
http.request.method
0          149423
GET          6676
POST          267
TRACE         252
OPTIONS         1
Name: count, dtype: int64
http.request.version
http.request.version
0                                                          149423
HTTP/1.1                                                     6493
HTTP/1.0                                                      698
Src=javascript:alert('Vulnerable')><Img Src=\" HTTP/1.1         2
/etc/passwd|?data=Download HTTP/1.1                             1
-a HTTP/

Final Clean

In [128]:
#drop the two ARP fields which are mostly empty and have many unique values
drop_cols = [
    "arp.dst.proto_ipv4",
    "arp.src.proto_ipv4"
]
df_clean = df_clean.drop(
    columns=drop_cols
)

#drop http.request.version, since the attack field was written in(data leakage)
df_clean = df_clean.drop(
    columns=["http.request.version"]
)

In [129]:
#X and y split
X = df_clean.drop(
    ["Attack_label", "Attack_type"],
    axis=1
)

y = df_clean["Attack_label"]

Output cleaned data 

In [130]:
df_clean.to_csv(
    "../data/processed/edgeiiot_clean_v1.csv",
    index=False
)

print("Saved Successfully!")

Saved Successfully!


In [131]:
df_saved = pd.read_csv(
    "../data/processed/edgeiiot_clean_v1.csv"
)
#data shape comparison
print("Shape of saved DataFrame:", df_saved.shape)
print("Shape of saved DataFrame:", df_clean.shape)

Shape of saved DataFrame: (156619, 42)
Shape of saved DataFrame: (156619, 42)


C:\Users\12985709\AppData\Local\Temp\ipykernel_29832\951687330.py:1: DtypeWarning: Columns (6,29,36,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df_saved = pd.read_csv(


In [107]:
object_cat = df_clean.select_dtypes(
    include=["object"]
).columns.tolist()

print(object_cat)

['arp.dst.proto_ipv4', 'arp.src.proto_ipv4', 'http.request.method', 'mqtt.conack.flags', 'mqtt.protoname', 'mqtt.topic', 'Attack_type']
